In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# 1. Load Data

In [ ]:
# Load DealScan dataset (Update with actual file path)
dealscan_file = "2021jan_2024sept.csv"
dealscan_df = pd.read_csv(dealscan_file, low_memory=False)
print(list(dealscan_df.columns))

call_report_file = "/Users/twylazhang/Desktop/Bank_Research/Data/Processed/CallR/callreport_panel_2021_2023.csv"
print(list(call_report_file.columns))

### The idrssd is different from the lender id inside the dealscan, need to use name to match

In [ ]:
# # Check data types and unique ID examples
# print("DealScan Lender_Id Sample:")
# print(dealscan_df["Lender_Id"].dropna().astype(str).str[:5].value_counts())

# print("\nCall Report idrssd Sample:")
# print(call_report_df["idrssd"].dropna().astype(str).str[:5].value_counts())

# # Check if Lender_Id values appear in idrssd
# common_ids = set(dealscan_df["Lender_Id"].dropna().astype(int)) & set(call_report_df["idrssd"].dropna().astype(int))
# print(f"\nNumber of matching IDs: {len(common_ids)}")

### re-process the call report to include names and load. After running first to acquire the files, it is commented to save running time

In [ ]:
# # Correct base directory for Call Report raw data
# base_directory = "/Users/twylazhang/Desktop/Bank_Research/Data/Raw/CallReport"

# # ✅ Ensure output directory exists
# output_dir = "/Users/twylazhang/Desktop/Bank_Research/DealScan-third-dataset/CallReport_dealscan"
# os.makedirs(output_dir, exist_ok=True)

# # Years to process
# years = [2020, 2021, 2022]

# for year in years:
#     print(f"Processing year {year}")

#     # Correct file path
#     directory = os.path.join(base_directory, f"FFIEC CDR Call Bulk All Schedules 1231{year}")
#     files = glob.glob(os.path.join(directory, "*.txt"))

#     call_report_data_vars = pd.DataFrame()

#     if not files:
#         print(f"No files found for year {year} in {directory}.")
#         continue

#     for i, fn in enumerate(files):
#         print(f"Reading file {i+1}/{len(files)}: {fn}")
#         temp_df = pd.read_csv(fn, delimiter='\t', low_memory=False, on_bad_lines='skip')

#         temp_df.columns = temp_df.columns.str.strip().str.lower()

#         if 'idrssd' not in temp_df.columns:
#             print(f"IDRSSD not found in {fn}, skipping file.")
#             continue

#         temp_df['idrssd'] = pd.to_numeric(temp_df['idrssd'], errors='coerce').astype('Int64')

#         # ✅ Ensure 'rssd9017' (Bank Name) is included
#         if 'rssd9017' in temp_df.columns:
#             temp_df.rename(columns={'rssd9017': 'bank_name'}, inplace=True)
#         else:
#             print(f"Warning: 'rssd9017' (Bank Name) column not found in {fn}")

#         # Merge datasets
#         if call_report_data_vars.empty:
#             call_report_data_vars = temp_df
#         else:
#             try:
#                 call_report_data_vars = call_report_data_vars.merge(temp_df, on='idrssd', how='left', suffixes=('', '.y'))
#             except KeyError as e:
#                 print(f"Error merging {fn}: {e}")

#     # ✅ Save the merged result to the correct directory
#     output_path = os.path.join(output_dir, f"CallReport_merged_vars_{year}.csv")
#     call_report_data_vars.to_csv(output_path, index=False)
#     print(f"Data for year {year} saved to {output_path}")

# print("Processing completed.")

Step 2: Modify Section 2B to Load and Save Data to the Correct Directory

In [ ]:
# for year in years:
#     print(f"Processing year {year}")

#     file_path_bs = os.path.join(output_dir, f"CallReport_merged_vars_{year}.csv")

#     # ✅ Skip missing files instead of crashing
#     if not os.path.exists(file_path_bs):
#         print(f"Warning: {file_path_bs} not found, skipping this year.")
#         continue

#     df = pd.read_csv(file_path_bs, low_memory=False)

#     df_s = pd.DataFrame()

#     # ✅ Drop rows where 'idrssd' is NaN
#     df = df.dropna(subset=['idrssd'])

#     # ✅ Convert 'idrssd' safely
#     df['idrssd'] = df['idrssd'].astype(int).astype(str)

#     df_s['idrssd'] = df['idrssd']
#     df_s['year'] = year

#     # ✅ Keep Bank Name
#     if 'bank_name' in df.columns:
#         df_s['bank_name'] = df['bank_name']
#     else:
#         print(f"Warning: 'bank_name' column missing for year {year}")

#     # Financial variables
#     df_s['assets'] = df['rcfd2170'].fillna(df['rcon2170'])
#     df_s['tier1_capital'] = df['rcfa8274'].fillna(df['rcoa8274'])
#     df_s['total_loans'] = df['rcfd5369'].fillna(df['rcon5369'])

#     df_s2 = df_s.add_suffix('_c')
#     df_s2.rename(columns={'year_c': 'year', 'idrssd_c': 'idrssd', 'bank_name_c': 'bank_name'}, inplace=True)
#     df_list.append(df_s2)

# df_panel = pd.concat(df_list, ignore_index=True)

# df_panel['year'] = pd.to_numeric(df_panel['year']) + 1
# df_panel = df_panel.dropna(subset=['idrssd'])  # ✅ Drop NaN values to avoid conversion errors

# # ✅ Save final Call Report dataset in the correct directory
# final_output_path = os.path.join(output_dir, "callreport_panel_2021_2023.csv")
# df_panel.to_csv(final_output_path, index=False)

# print(f"Final Call Report file saved to {final_output_path}")

In [ ]:
call_report_df =  pd.read_csv("CallReport_dealscan/callreport_panel_2021_2023.csv", low_memory=False)
print(list(call_report_df.columns))

# 2. compare

In [ ]:
### STEP 2: FILTER FOR U.S. BANKS IN DEALSCAN ###
print("Filtering U.S. banks from DealScan...")
us_dealscan_df = dealscan_df[dealscan_df["Lender_Operating_Country"] == "United States"]

# Count unique U.S. lenders in DealScan
num_unique_lenders = us_dealscan_df["Lender_Name"].nunique()
print(f"Number of unique U.S. lenders in DealScan: {num_unique_lenders}")
us_dealscan_df.to_csv("usBanks_dealscan_df.csv")

In [ ]:
### STEP 3: CLEAN AND MATCH BANK NAMES ###
def clean_bank_name(name):
    """ Function to clean and standardize bank names for better matching """
    if pd.isna(name):
        return None
    name = str(name).strip().lower()
    name = name.replace("bank", "").replace("inc", "").replace("corporation", "")
    name = name.replace("ltd", "").replace("llc", "").replace("co.", "").replace(",", "")
    return name

# Apply name cleaning
us_dealscan_df["Lender_Clean"] = us_dealscan_df["Lender_Name"].apply(clean_bank_name)
call_report_df["Bank_Clean"] = call_report_df["bank_name"].apply(clean_bank_name)

# Merge datasets to find U.S. banks in both DealScan and Call Report
print("Matching U.S. banks between DealScan and Call Report...")
matched_banks = us_dealscan_df.merge(call_report_df, left_on="Lender_Clean", right_on="Bank_Clean", how="inner")

# Count matched U.S. banks
num_matched_banks = matched_banks["Lender_Name"].nunique()
print(f"Number of U.S. banks in DealScan that match Call Report: {num_matched_banks}")

In [ ]:
### STEP 4: CHECK IF LARGE BANKS DOMINATE ###
# Count occurrences of each lender in the full DealScan dataset
bank_loan_counts = dealscan_df.groupby(["Lender_Name", "Lender_Operating_Country"]).size().reset_index(name="Loan_Count")

# Identify the total number of banks in DealScan
total_banks_in_dealscan = dealscan_df["Lender_Name"].nunique()

# Identify the number of U.S. banks in DealScan
us_banks_in_dealscan = dealscan_df[dealscan_df["Lender_Operating_Country"] == "United States"]["Lender_Name"].nunique()

# Calculate the percentage of U.S. banks in the entire DealScan dataset
us_bank_percentage_in_dealscan = (us_banks_in_dealscan / total_banks_in_dealscan) * 100

# Identify the top 10 most active banks in DealScan
top_10_banks = bank_loan_counts.sort_values(by="Loan_Count", ascending=False).head(10)

print(f"\nTotal number of banks in DealScan: {total_banks_in_dealscan}")
print(f"Total number of U.S. banks in DealScan: {us_banks_in_dealscan}")
print(f"Percentage of U.S. banks in DealScan: {us_bank_percentage_in_dealscan:.2f}%\n")

print("\nTop 10 most active banks in DealScan (with country):")
print(top_10_banks)

# Check lead arranger dominance in the full dataset
if "Lead_Arranger" in dealscan_df.columns:
    lead_arrangers = dealscan_df.groupby(["Lead_Arranger", "Lender_Operating_Country"])["Lender_Name"].count().reset_index()
    lead_arrangers = lead_arrangers.sort_values(by="Lender_Name", ascending=False).head(10)

    print("\nTop Lead Arrangers in DealScan (with country):")
    print(lead_arrangers)
else:
    print("\nNo 'Lead Arranger' column found in dataset. Skipping lead arranger analysis.")

### STEP 5: SAVE RESULTS ###
print("\nSaving results...")
top_10_banks.to_csv("top_10_banks_in_dealscan.csv", index=False)
lead_arrangers.to_csv("top_lead_arrangers_in_dealscan.csv", index=False)

# Save the U.S. bank percentage to a text file
with open("us_bank_percentage_in_dealscan.txt", "w") as file:
    file.write(f"Total number of banks in DealScan: {total_banks_in_dealscan}\n")
    file.write(f"Total number of U.S. banks in DealScan: {us_banks_in_dealscan}\n")
    file.write(f"Percentage of U.S. banks in DealScan: {us_bank_percentage_in_dealscan:.2f}%\n")

print("\nAnalysis complete. Check the output files for details.")